In [ ]:
# SPDX-FileCopyrightText: 2026 Mario Gemoll
# SPDX-License-Identifier: 0BSD

import os
import subprocess

REMOTES = (
    "https://github.com/mariogemoll/pick-and-place.git",
    "git@github.com:mariogemoll/pick-and-place.git",
)


def in_checkout() -> bool:
    try:
        origin = subprocess.run(
            ["git", "remote", "get-url", "origin"],
            capture_output=True,
            text=True,
            check=True,
        )
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False
    return origin.stdout.strip() in REMOTES


if not in_checkout():
    !git clone --recurse-submodules https://github.com/mariogemoll/pick-and-place.git

if not os.getcwd().endswith("pick-and-place/py/notebooks"):
    %cd pick-and-place/py/notebooks

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PAP_DATA_ROOT", str(Path.home() / "pick-and-place-data"))

# Colab has none of this; a local checkout installed per the README has all of it.
if "google.colab" in sys.modules:
    !sudo apt-get -qq install -y libegl1 libgl1 libopengl0
    %pip install -q -e ..
    !python -m pick_and_place.cli.pap render-apriltag-textures --all-defaults

# The bakeoff

The scripted expert, ACT and the flow-matching policy, scored against the same
domain-randomized scenarios and compared.

## Running it

1. Set `ACT_CHECKPOINT` and `FLOW_REPOSITORY` below. Either takes a Hugging Face
   repository id or a local path; an arm without one is skipped.
2. Set `RUN_EVALUATIONS = True` to score the arms, or leave it `False` to read
   runs already under `RUNS`.

Scoring costs about a minute per scenario per core. `SCENARIOS` defaults to a
six-scenario slice; the last cell has the recipe for the full suite.

In [ ]:
import json
import lzma
from dataclasses import dataclass, field

REPO = Path.cwd().parents[1]
SUITE = REPO / "config" / "evaluation" / "dr_100_v1.json.xz"
DATA_ROOT = Path(os.environ.get("PAP_DATA_ROOT", Path.home() / "pick-and-place-data"))
RUNS = DATA_ROOT / "evaluations" / "bakeoff-dr"

SCENARIOS = 6              # scenarios per arm; None runs the whole suite
ACT_CHECKPOINT = None      # e.g. "mariogemoll/pap-act-randomized"
FLOW_REPOSITORY = None     # a repo holding checkpoint-*.pt and an export/ directory

RUN_EVALUATIONS = False    # True scores the arms; False reads runs already in RUNS
SAVE_VIDEOS = True         # the policy's own camera frames, two files a scenario

## The suite

In [ ]:
manifest = json.loads(lzma.open(SUITE).read())
scenarios = manifest["scenarios"]

suite_name = manifest["suite"]
schema = manifest["schema_version"]
print(f"{suite_name}: {len(scenarios)} scenarios, schema v{schema}")
print("randomization preset:", {s["domain_randomization_preset"] for s in scenarios})
print("workspace regions:  ", sorted({s["workspace_region"] for s in scenarios}))

first = scenarios[0]
print(f"\n{first['scenario_id']} draws, for example:")
draw = first["domain_randomization_sample"]
print("  light intensity ", round(draw["light_intensity"], 3))
print("  cube at         ", [round(v, 3) for v in first["source_position_m"]])
print("  target at       ", [round(v, 3) for v in first["target_position_m"]])
print("  physics keys    ", sorted(first["physics_sample"])[:6], "...")

## The arms

In [ ]:
@dataclass
class Arm:
    """A controller leaf, its flags, and the colour it is drawn in."""

    leaf: str
    color: str
    flags: list[str] = field(default_factory=list)
    available: bool = True


def flow_checkpoint_flags(repository: str) -> list[str]:
    """Resolve a flow-policy repository to the two paths the leaf wants."""
    from huggingface_hub import snapshot_download

    local = Path(snapshot_download(repo_id=repository))
    checkpoint = sorted(local.glob("checkpoint*.pt"))[-1]
    return ["--checkpoint", str(checkpoint), "--flow-export", str(local / "export")]


ARMS = {
    "scripted": Arm(leaf="scripted", color="#2a78d6"),
    "act": Arm(
        leaf="lerobot",
        color="#eb6834",
        flags=["--checkpoint", str(ACT_CHECKPOINT)] if ACT_CHECKPOINT else [],
        available=ACT_CHECKPOINT is not None,
    ),
    "flow": Arm(
        leaf="flow-image",
        color="#1baf7a",
        flags=flow_checkpoint_flags(FLOW_REPOSITORY) if FLOW_REPOSITORY else [],
        available=FLOW_REPOSITORY is not None,
    ),
}


def eval_command(name: str, arm: Arm) -> list[str]:
    command = [
        "pap", "eval-policy-sim", arm.leaf,
        "--manifest", str(SUITE),
        "--output", str(RUNS / name),
    ]
    if SAVE_VIDEOS:
        command.append("--save-videos")
    if SCENARIOS is not None:
        command += ["--limit", str(SCENARIOS)]
    return command + arm.flags


for name, arm in ARMS.items():
    mark = " " if arm.available else "  (skipped: no checkpoint)"
    print(f"{name}:{mark}\n  " + " ".join(eval_command(name, arm)) + "\n")

In [ ]:
import re

from tqdm.auto import tqdm

from pick_and_place.cli.eval_policy_sim_parser import build_parser, validate
from pick_and_place.rollout.evaluation import EvaluationRun as EvaluationConfig
from pick_and_place.rollout.evaluation import run_evaluation

PROGRESS = re.compile(r"^\[(\d+)/(\d+)\]")


def score(name: str, arm: Arm) -> dict:
    # Parses the same argv the cell above prints, so the two cannot diverge.
    parser = build_parser()
    args = parser.parse_args(eval_command(name, arm)[2:])
    validate(parser, args)

    bar = tqdm(desc=name, unit="scenario", leave=True)

    def report(line: str) -> None:
        tqdm.write(line)
        progress = PROGRESS.match(line)
        if progress is not None:
            bar.total = int(progress.group(2))
            bar.n = int(progress.group(1))
            bar.refresh()

    try:
        return run_evaluation(EvaluationConfig.from_args(args), report=report)
    finally:
        bar.close()

In [ ]:
if not RUN_EVALUATIONS:
    print("RUN_EVALUATIONS is False -- reading whatever is already under")
    print(RUNS)

for name, arm in ARMS.items() if RUN_EVALUATIONS else ():
    if not arm.available:
        continue
    if (RUNS / name / "run.json").exists():
        print(f"{name}: already scored, leaving it alone")
        continue
    summary = score(name, arm)
    print(f"{name}: {summary['success_count']}/{summary['episode_count']} successes")

## Load the runs

In [ ]:
from pick_and_place.cli.compare_policy_evaluations import (
    METRICS,
    PLACED_TOLERANCE_M,
    EvaluationRun,
    mcnemar,
    wilson_interval,
)

runs = {}
for name in ARMS:
    directory = RUNS / name
    if (directory / "run.json").exists() or list(directory.glob("shard-*/run.json")):
        runs[name] = EvaluationRun(directory)

if not runs:
    raise SystemExit(
        f"No evaluation runs under {RUNS}.\n"
        "Set RUN_EVALUATIONS = True above, or point RUNS at runs you already have."
    )

for name, run in runs.items():
    suite = run.run["scenario_manifest"]
    world = run.run["environment"]
    size = f"{world['image_width']}x{world['image_height']}"
    print(f"{name:<9} {len(run.episodes):>3} scenarios  "
          f"{suite['suite']}  manifest {suite['sha256'][:12]}  {size}")

shared = sorted(set.intersection(*(set(run.scenario_ids) for run in runs.values())))
print(f"\n{len(shared)} scenarios common to every arm.")

## Compare

In [ ]:
present = [str(RUNS / name) for name in runs]
print(subprocess.run(
    ["pap", "compare-policy-evaluations", *present],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout)

### Rates, with Wilson intervals

In [ ]:
import pandas as pd

rows = []
for name, run in runs.items():
    row = {"arm": name, "n": len(run.episodes)}
    for metric in METRICS:
        outcomes = run.outcomes(metric)
        count = sum(outcomes.values())
        low, high = wilson_interval(count, len(outcomes))
        row[metric] = f"{count}/{len(outcomes)} = {count / len(outcomes):.0%}"
        row[f"{metric} 95% CI"] = f"[{low:.0%}, {high:.0%}]"
    rows.append(row)

headline = pd.DataFrame(rows).set_index("arm")
headline

## Paired comparison

In [ ]:
baseline_name = "scripted" if "scripted" in runs else next(iter(runs))
baseline = runs[baseline_name]

comparisons = []
for name, run in runs.items():
    if name == baseline_name:
        continue
    test = mcnemar(run.outcomes("success"), baseline.outcomes("success"), shared)
    comparisons.append({
        "arm": name,
        f"beat {baseline_name} on": test["only_first"],
        f"lost to {baseline_name} on": test["only_second"],
        "both": test["both"],
        "neither": test["neither"],
        "p (exact)": round(test["p_value_exact"], 4),
    })

if not comparisons:
    print(f"Only {baseline_name} has a run; nothing to pair it against.")

pd.DataFrame(comparisons).set_index("arm") if comparisons else None

## Milestones

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c9c8c3",
    "axes.labelcolor": "#3d3d38",
    "axes.axisbelow": True,
    "text.color": "#1a1a19",
    "xtick.color": "#6b6a63",
    "ytick.color": "#6b6a63",
    "grid.color": "#e8e7e2",
    "grid.linewidth": 0.8,
    "legend.frameon": False,
})

MILESTONES = [
    "pickup_contact_attempted",
    "cube_lifted",
    "stable_carry",
    "target_reached_while_holding",
    "cube_released",
    "cube_settled",
    "successful_placement",
]


def milestone_rate(run, milestone):
    return sum(e["milestones"][milestone] for e in run.episodes) / len(run.episodes)


milestones = pd.DataFrame(
    {name: [milestone_rate(run, m) for m in MILESTONES] for name, run in runs.items()},
    index=[m.replace("_", " ") for m in MILESTONES],
)

positions = range(len(MILESTONES))
height = 0.8 / len(runs)
figure, axes = plt.subplots(figsize=(7.5, 4.2))
for index, (name, run) in enumerate(runs.items()):
    offset = (index - (len(runs) - 1) / 2) * height
    bars = axes.barh(
        [p + offset for p in positions], milestones[name], height=height * 0.9,
        color=ARMS[name].color, label=name,
    )
    axes.bar_label(bars, labels=[f"{v:.0%}" for v in milestones[name]],
                   padding=3, fontsize=8, color="#6b6a63")

axes.set_yticks(list(positions), milestones.index)
axes.invert_yaxis()
axes.set_xlim(0, 1.12)
axes.set_xlabel("share of scenarios reaching this milestone")
axes.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes.grid(axis="x")
axes.grid(axis="y", visible=False)
if len(runs) > 1:
    axes.legend(loc="lower right")
axes.set_title("Which milestones each arm reached", loc="left", pad=12)
plt.tight_layout()
plt.show()

milestones.style.format("{:.0%}")

## Failures

In [ ]:
FAILURES = [
    "missed_pickup",
    "unstable_or_lost_grasp",
    "early_release",
    "off_target_placement",
    "unexpected_collision",
    "cube_out_of_bounds",
    "timeout",
]

taxonomy = pd.DataFrame(
    {
        name: [sum(e["failures"][f] for e in run.episodes) for f in FAILURES]
        for name, run in runs.items()
    },
    index=[f.replace("_", " ") for f in FAILURES],
)
taxonomy.loc["episodes"] = [len(run.episodes) for run in runs.values()]
taxonomy

## Placement error

In [ ]:
figure, axes = plt.subplots(figsize=(7.5, 3.8))
for name, run in runs.items():
    errors = sorted(e["final_xy_error_m"] * 100 for e in run.episodes)
    share = [(index + 1) / len(errors) for index in range(len(errors))]
    axes.step(
        [0, *errors], [0, *share], where="post",
        color=ARMS[name].color, linewidth=2, label=name,
    )

axes.axvline(4.0, color="#6b6a63", linewidth=1, linestyle=":")
axes.annotate("4 cm: success", (4.0, 0.04), xytext=(-6, 0), textcoords="offset points",
              ha="right", fontsize=8, color="#6b6a63")
axes.axvline(PLACED_TOLERANCE_M * 100, color="#6b6a63", linewidth=1, linestyle=":")
axes.annotate("6 cm: placed", (PLACED_TOLERANCE_M * 100, 0.14), xytext=(6, 0),
              textcoords="offset points", fontsize=8, color="#6b6a63")

axes.set_xlabel("final cube-to-target distance (cm)")
axes.set_ylabel("share of scenarios")
axes.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes.set_ylim(0, 1.02)
axes.grid(axis="y")
if len(runs) > 1:
    axes.legend(loc="lower right")
axes.set_title("Where the cube ended up", loc="left", pad=12)
plt.tight_layout()
plt.show()

## A disagreement

In [ ]:
from IPython.display import Video, display

interesting = None
if len(runs) == 1:
    print(f"Only {baseline_name} ran, so there is nothing to disagree with.")

for name, run in runs.items():
    if name == baseline_name:
        continue
    test = mcnemar(run.outcomes("success"), baseline.outcomes("success"), shared)
    lost = test["only_second_scenarios"]
    won = test["only_first_scenarios"]
    print(f"{name} failed where {baseline_name} succeeded: {lost}")
    print(f"{name} succeeded where {baseline_name} failed: {won}\n")
    interesting = interesting or next(iter(test["only_second_scenarios"]), None)

if interesting is None:
    worst = max(baseline.episodes, key=lambda e: e["final_xy_error_m"])
    interesting = worst["scenario_id"]
    print(f"Showing {baseline_name}'s worst placement instead.")

print(f"scenario: {interesting}")
for name, run in runs.items():
    video = RUNS / name / "videos" / f"{interesting}-overhead.mp4"
    if video.exists():
        print(f"\n{name} -- overhead")
        display(Video(str(video), embed=True, width=420))

## The full suite

```sh
# one shard per worker, disjoint slices of the same suite
for offset in 0 25 50 75; do
  pap eval-policy-sim flow-image \
    --manifest config/evaluation/dr_100_v1.json.xz \
    --checkpoint "$CHECKPOINT" --flow-export "$EXPORT" \
    --offset "$offset" --limit 25 \
    --output "$PAP_DATA_ROOT/evaluations/bakeoff-dr/flow/shard-$offset" &
done
wait

pap compare-policy-evaluations \
  "$PAP_DATA_ROOT"/evaluations/bakeoff-dr/{scripted,act,flow} \
  --baseline "$PAP_DATA_ROOT/evaluations/bakeoff-dr/scripted"
```

`EvaluationRun` picks up `shard-*/run.json` on its own. `canonical_100_v1` is the
same suite with randomization off, and `heldout_256_v1` is the untouched one.